In [12]:
import os
from nats_bench import create
import pprint
import torch

# Set the TORCH_HOME environment variable for NATS-Bench files
os.environ['TORCH_HOME'] = "./.torch/"

In [18]:
# Create the API object for NATS-Bench
api = create(None, 'sss', fast_mode=True, verbose=False)

param_max=0.105
hp ='90'

# Find the best model index for the given constraints
index, acc = api.find_best(dataset='cifar10', metric_on_set='test', param_max=param_max, hp =hp)

print('Best model index: ', index)
print('Best model test accuracy: ', acc)

# Get more information about the best model and its cost
more_info = api.get_more_info(index, 'cifar10', hp=hp)
cost_info = api.get_cost_info(index, 'cifar10', hp=hp)
pprint.pprint('Cifar 10 dataset, model with the highest accuracy:' + str(more_info) + str(cost_info))

[2026-02-22 19:21:39] Try to use the default NATS-Bench (size) path from fast_mode=True and path=None.
Best model index:  4881
Best model test accuracy:  90.54
("Cifar 10 dataset, model with the highest accuracy:{'train-loss': "
 "0.11857536593914032, 'train-accuracy': 96.108, 'train-per-time': "
 "6.426358461380005, 'train-all-time': 578.3722615242004, 'comment': 'In this "
 'dict, train-loss/accuracy/time is the metric on the train+valid sets of '
 'CIFAR-10. The test-loss/accuracy/time is the performance of the CIFAR-10 '
 'test set after training on the train+valid sets by 90 epochs. The per-time '
 "and total-time indicate the per epoch and total time costs, respectively.', "
 "'test-loss': 0.3243563588142395, 'test-accuracy': 90.54, 'test-per-time': "
 "0.6860654354095459, 'test-all-time': 61.74588918685913}{'flops': 29.51185, "
 "'params': 0.102394, 'latency': 0.01589440672981496, 'T-train@epoch': "
 "6.426358461380005, 'T-train@total': 578.3722615242004, 'T-ori-test@epoch': "
 

In [19]:
from types import SimpleNamespace
from xautodl.models import get_cell_based_tiny_net
config = api.get_net_config(index, 'cifar10')
config = SimpleNamespace(**config)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = get_cell_based_tiny_net(config).to(DEVICE)

torch.save(model.state_dict(), "sample.pt")